In [0]:
import requests
import json

NOAA_TOKEN = "TDaUPHpXeUZSfwvMyehKnVxmwiVVgpMO"
STATION_ID = "GHCND:USC00413370"

# CHANGE: Target the data categories query parameter endpoint
url = f"https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes?stationid={STATION_ID}&limit=1000"

headers = {"token": NOAA_TOKEN}
response = requests.get(url, headers=headers)

# print(json.dumps(response.json(), indent=2))

data = response.json()

for category in data.get("results", []):
    print(
        category.get("id"),
        "->",
        category.get("name")
    )


In [0]:
import requests
import json
import time
from pyspark.sql.functions import current_timestamp

# Token required to access data from government website
NOAA_TOKEN = "TDaUPHpXeUZSfwvMyehKnVxmwiVVgpMO"
headers = {
    "token": NOAA_TOKEN
}
combined_records = []

# Each city's weather station has a unique ID we must reference to access the desired data
# There's 2 Frisco stations, one has data from 1966-2023 and the other has data from 1941-present but only for precipitation
# We take the temperature data from the former, and precipitation data from the latter.
# For present temperature data, we use DFW airport's weather as it's an adjacent city to make up for the gap
station_configs = [
    {
        "id": "GHCND:USC00413370",
        "label": "Frisco_Historical",
        "start": "2023-01-01",  # Example: pulling the final months of history
        "end": "2023-08-31"
    },
    {
        "id": "GHCND:US1TXDN0091",
        "label": "Frisco_Modern_Precip",
        "start": "2026-07-01",  # Recent weeks up to its latest data points
        "end": "2026-07-15"
    },
    {
        "id": "GHCND:USW00003927",
        "label": "DFW_Airport_Proxy",
        "start": "2023-09-01",  # Seamlessly picks up the day Frisco went dark
        "end": "2023-09-30"
    }
]

for config in station_configs:
    url = f"https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid={config['id']}&startdate={config['start']}&enddate={config['end']}&limit=1000"
    
    response = requests.get(url, headers=headers)
    raw_json = response.json()
    
    if "results" in raw_json:
        for record in raw_json["results"]:
            record["station_label"] = config["label"]
            combined_records.append(record)
            
    # Data Engineer Best Practice: Sleep for 0.5 seconds between loops 
    # to strictly respect NOAA's rate limit of 5 requests per second.
    time.sleep(0.5)

# Build a clean DataFrame combining all target windows
df_bronze_noaa = spark.createDataFrame(combined_records)

df_bronze_noaa = df_bronze_noaa.select(
    "date",
    "datatype",
    "value",
    "station",
    "station_label"
).withColumn("ingested_at", current_timestamp())

display(df_bronze_noaa)

In [0]:
df_bronze_noaa.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("weather_project.frisco_weather.bronze_noaa_historical")